In [1]:
# Импортируем алхимию и создаем соединение
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
POSTGRES_HOST= os.getenv('POSTGRES_HOST', 'localhost')

conn = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:5432/{POSTGRES_DB}')


In [2]:
sql = '''
    create table if not exists staging.candles(
    datetime TIMESTAMPTZ NOT NULL,
    figi VARCHAR(255) NOT NULL,
    interval VARCHAR(255) NOT NULL,
    open DECIMAL(12,2),
    close DECIMAL(12,2),
    high DECIMAL(12,2),
    low DECIMAL(12,2),
    is_complete BOOLEAN,
    candle_source VARCHAR(255),
    volume BIGINT,
    volume_buy BIGINT,
    volume_sell BIGINT,
    source_job_id INTEGER,
    loaded_at TIMESTAMPTZ DEFAULT NOW()
    )
'''

with conn.begin() as c:
    c.execute(text(sql))

In [ ]:
sql = '''
    WITH raw_data AS (
    SELECT
        jsonb_array_elements(response_data -> 'candles') AS candle,
        request_payload #>> '{figi}' AS figi,
        request_payload #>> '{interval}' AS interval,
        id AS source_job_id
    FROM raw.extract_api_jobs
    WHERE endpoint = 'MarketDataService/GetCandles'
      AND status_code = 200
      AND 
    )
    SELECT
        (candle ->> 'time')::TIMESTAMPTZ AS datetime,
        figi,
        interval,
        ((candle -> 'open' ->> 'units')::DECIMAL + (candle -> 'open' ->> 'nano')::DECIMAL / 1e9) AS open,
        ((candle -> 'close' ->> 'units')::DECIMAL + (candle -> 'close' ->> 'nano')::DECIMAL / 1e9) AS close,
        ((candle -> 'high' ->> 'units')::DECIMAL + (candle -> 'high' ->> 'nano')::DECIMAL / 1e9) AS high,
        ((candle -> 'low' ->> 'units')::DECIMAL + (candle -> 'low' ->> 'nano')::DECIMAL / 1e9) AS low,
        (candle ->> 'isComplete')::BOOLEAN AS is_complete,
        candle ->> 'candleSource' AS candle_source,
        (candle ->> 'volume')::BIGINT AS volume,
        (candle ->> 'volumeBuy')::BIGINT AS volume_buy,
        (candle ->> 'volumeSell')::BIGINT AS volume_sell,
        source_job_id,
        CURRENT_TIMESTAMP AS loaded_at
    FROM raw_data
'''

df = pd.read_sql(sql, conn)
df

,datetime,figi,interval,open,close,high,low,is_complete,candle_source,volume,volume_buy,volume_sell,source_job_id,loaded_at
0,2026-06-29 06:00:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,300.38,298.58,300.38,298.23,True,CANDLE_SOURCE_EXCHANGE,783801,303457,480344,1,2026-07-13 06:56:30.851353+00:00
1,2026-06-29 06:15:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,298.58,297.91,298.58,297.26,True,CANDLE_SOURCE_EXCHANGE,721745,324292,397453,1,2026-07-13 06:56:30.851353+00:00
2,2026-06-29 06:30:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,297.91,297.66,298.04,297.00,True,CANDLE_SOURCE_EXCHANGE,366784,159591,207193,1,2026-07-13 06:56:30.851353+00:00
3,2026-06-29 06:45:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,297.66,297.50,298.23,297.10,True,CANDLE_SOURCE_EXCHANGE,413727,179614,234113,1,2026-07-13 06:56:30.851353+00:00
4,2026-06-29 07:00:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,297.49,297.71,297.88,297.07,True,CANDLE_SOURCE_EXCHANGE,1054615,416568,638047,1,2026-07-13 06:56:30.851353+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1503,2026-07-12 14:00:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,293.05,292.80,293.10,292.70,True,CANDLE_SOURCE_EXCHANGE,96147,30577,65570,32,2026-07-13 06:56:30.851353+00:00
1504,2026-07-12 14:15:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,292.76,292.57,292.81,292.57,True,CANDLE_SOURCE_EXCHANGE,64110,20517,43593,32,2026-07-13 06:56:30.851353+00:00
1505,2026-07-12 14:30:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,292.58,292.23,292.58,292.04,True,CANDLE_SOURCE_EXCHANGE,92030,20113,71917,32,2026-07-13 06:56:30.851353+00:00
1506,2026-07-12 14:45:00+00:00,BBG004730N88,CANDLE_INTERVAL_15_MIN,292.25,292.42,292.45,292.24,True,CANDLE_SOURCE_EXCHANGE,27240,16530,10710,32,2026-07-13 06:56:30.851353+00:00


In [ ]:
sql = '''
    SELECT
        jsonb_array_elements(response_data -> 'candles') AS candle,
        request_payload #>> '{figi}' AS figi,
        request_payload #>> '{interval}' AS interval,
        id AS source_job_id
    FROM raw.extract_api_jobs
    WHERE endpoint = 'MarketDataService/GetCandles'
      AND status_code = 200
'''

df = pd.read_sql(sql, conn)
df

,candle,figi,interval,source_job_id
0,"{'low': {'nano': 230000000, 'units': '298'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,1
1,"{'low': {'nano': 260000000, 'units': '297'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,1
2,"{'low': {'nano': 0, 'units': '297'}, 'high': {...",BBG004730N88,CANDLE_INTERVAL_15_MIN,1
3,"{'low': {'nano': 100000000, 'units': '297'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,1
4,"{'low': {'nano': 70000000, 'units': '297'}, 'h...",BBG004730N88,CANDLE_INTERVAL_15_MIN,1
...,...,...,...,...
1503,"{'low': {'nano': 700000000, 'units': '292'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,32
1504,"{'low': {'nano': 570000000, 'units': '292'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,32
1505,"{'low': {'nano': 40000000, 'units': '292'}, 'h...",BBG004730N88,CANDLE_INTERVAL_15_MIN,32
1506,"{'low': {'nano': 240000000, 'units': '292'}, '...",BBG004730N88,CANDLE_INTERVAL_15_MIN,32


In [9]:
sql = '''
    select *
    from raw.extract_api_jobs
    where endpoint = 'MarketDataService/GetCandles'
      and status_code = 200
      and loaded_at >= (select coalesce(max(loaded_at)::date - interval '1 day', date_trunc('month', current_date)) from staging.candles)
'''

df = pd.read_sql(sql, conn)
df

,id,endpoint,request_payload,response_data,status_code,loaded_at
0,3,MarketDataService/GetCandles,"{'to': '2026-07-01T05:15:45.774673+00:00', 'fi...","{'candles': [{'low': {'nano': 890000000, 'unit...",200,2026-07-01 05:15:45.932152+00:00
1,4,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:47.038426+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:51.891103+00:00
2,5,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:49.436653+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:54.170052+00:00
3,6,MarketDataService/GetCandles,"{'to': '2026-07-03T05:46:40.445665+00:00', 'fi...","{'candles': [{'low': {'nano': 670000000, 'unit...",200,2026-07-03 05:46:44.184541+00:00
4,7,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:12.709434+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:15.254333+00:00
5,8,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:30.390869+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:32.991671+00:00
6,12,MarketDataService/GetCandles,"{'to': '2026-07-04T08:30:53.080675+00:00', 'fi...","{'candles': [{'low': {'nano': 700000000, 'unit...",200,2026-07-04 08:30:58.814825+00:00
7,13,MarketDataService/GetCandles,"{'to': '2026-07-04T08:32:15.115472+00:00', 'fi...","{'candles': [{'low': {'nano': 700000000, 'unit...",200,2026-07-04 08:32:18.213452+00:00
8,15,MarketDataService/GetCandles,"{'to': '2026-07-05T13:29:38.399146+00:00', 'fi...","{'candles': [{'low': {'nano': 510000000, 'unit...",200,2026-07-05 13:29:44.256118+00:00
9,16,MarketDataService/GetCandles,"{'to': '2026-07-07T05:21:07.925461+00:00', 'fi...","{'candles': [{'low': {'nano': 230000000, 'unit...",200,2026-07-07 05:21:14.673027+00:00
